# PyDI Data Integration Workflow: Products

This notebook demonstrates how PyDI is used for end-to-end data integration. We'll work with product datasets to showcase the data integration pipeline from information extraction over schema and entity matching to data fusion.

## Table of Contents

* [Part 1: LLM-Based Schema Matching](#Part-1:-LLM-Based-Schema-Matching)
    * [Step 1: Schema Matching](#Step-1:-Schema-Matching)
    * [Step 2: Data profiling (again)](#Step-2:-Data-profiling-(again))
* [Part 2: Entity Matching](#Part-2:-Entity-Matching)
    * [Step 1: Blocking](#Step-1:-Blocking)
    * [Step 2: Evaluate Blocking Against Ground Truth](#Step-2:-Evaluate-Blocking-Against-Ground-Truth)
    * [Step 3: Entity Matching with Comparators](#Step-3:-Entity-Matching-with-Comparators)
    * [Step 4: Rule Based Matcher](#Step-4:-Rule-Based-Matcher)
    * [Step 5: Evaluate Matching Against Ground Truth](#Step-5:-Evaluate-Matching-Against-Ground-Truth)
    * [Step 6: GreedyOneToOneMatchingAlgorithm](#Step-6:-GreedyOneToOneMatchingAlgorithm)
    * [Step 7: MaximumBipartiteMatching](#Step-7:-MaximumBipartiteMatching)
    * [Step 8: ML-Based Matcher](#Step-8:-ML-Based-Matcher)
* [Part 3: Data Fusion](#Part-3:-Data-Fusion)
    * [Step 1: Define Fusion Strategy](#Step-1:-Define-Fusion-Strategy)
    * [Step 2: Run Fusion](#Step-2:-Run-Fusion)
    * [Step 3: Evaluate Data Fusion](#step-3-evaluate-data-fusion)

## Part 1: LLM-Based Schema Matching

### Step 1: Schema Matching 

In [1]:
import json
import pandas as pd
import logging
import os
import re
from PyDI.io import load_json
from PyDI.io import load_csv
from pathlib import Path
from langchain_openai import ChatOpenAI

from PyDI.schemamatching import LLMBasedSchemaMatcher, SchemaTranslator
from PyDI.utils import DataProfiler
from PyDI.normalization import load_normalization_spec
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.entitymatching import GreedyOneToOneMatchingAlgorithm
from PyDI.entitymatching import MaximumBipartiteMatching

from PyDI.entitymatching import MLBasedMatcher, FeatureExtractor
from sklearn.ensemble import RandomForestClassifier

from PyDI.fusion import DataFusionStrategy, longest_string, shortest_string, prefer_higher_trust, voting, maximum, minimum
from PyDI.fusion import DataFusionEngine
from PyDI.fusion import exact_match, numeric_tolerance_match
from PyDI.fusion import DataFusionEvaluator, DataFusionStrategy, exact_match, numeric_tolerance_match, tokenized_match


from dotenv import load_dotenv
load_dotenv()


# Paths relative to this notebook
NOTEBOOK_DIR = Path(".").resolve()
INPUT_DIR = NOTEBOOK_DIR / "input"
OUTPUT_DIR = NOTEBOOK_DIR / "output"
SCHEMA_DIR = INPUT_DIR / "schemamatching"

#load datasets
products_1_cleaned = load_json(INPUT_DIR / "data_cleaned_final" / "dataset_1_normalized.json")
products_1_cleaned.attrs["dataset_name"] = "products_1"

products_2_cleaned = load_json(INPUT_DIR / "data_cleaned_final" / "dataset_2_normalized.json")
products_2_cleaned.attrs["dataset_name"] = "products_2"

products_3_cleaned = load_json(INPUT_DIR / "data_cleaned_final" / "dataset_3_normalized.json")
products_3_cleaned.attrs["dataset_name"] = "products_3"

products_4_cleaned = load_json(INPUT_DIR / "data_cleaned_final" / "dataset_4_normalized.json")
products_4_cleaned.attrs["dataset_name"] = "products_4"

datasets = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]
names = ["products_1", "products_2", "products_3", "products_4"]

# Load schema
with open(SCHEMA_DIR / "products_target_schema.json") as f:
    target_schema = json.load(f)

spec = load_normalization_spec(SCHEMA_DIR / "products_target_schema.json")
target_columns = list(spec.columns.keys())

df_target = pd.DataFrame(columns=target_columns)
df_target.attrs["dataset_name"] = "target_schema"

# Match each dataset
matcher = LLMBasedSchemaMatcher(
    chat_model=ChatOpenAI(model="gpt-5.2", temperature=0),
    num_rows=40,
    target_schema=target_schema,
)

mappings = {}
translator = SchemaTranslator()

c:\Users\hussa\anaconda3\envs\hiwi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
datasets[0].attrs["dataset_name"] = "products_1"
mapping_1 = matcher.match(datasets[0], df_target)
df_final_1 = translator.translate(datasets[0], mapping=mapping_1, normalize=None)

datasets[1].attrs["dataset_name"] = "products_2"
mapping_2 = matcher.match(datasets[1], df_target)
df_final_2 = translator.translate(datasets[1], mapping=mapping_2, normalize=None)

datasets[2].attrs["dataset_name"] = "products_3"
mapping_3 = matcher.match(datasets[2], df_target)
df_final_3 = translator.translate(datasets[2], mapping=mapping_3, normalize=None)

datasets[3].attrs["dataset_name"] = "products_4"
mapping_4 = matcher.match(datasets[3], df_target)
df_final_4 = translator.translate(datasets[3], mapping=mapping_4, normalize=None)

### Step 2: Data profiling

In [3]:
profiling_datasets = [
    df_final_1,
    df_final_2,
    df_final_3,
    df_final_4
]

names = ["products_1", "products_2", "products_3", "products_4"]


total_records = sum(len(df) for df in profiling_datasets)
print(f"Total records across all profiling_datasets: {total_records:,}")

# Initialize the DataProfiler
profiler = DataProfiler()

for df, name in zip(profiling_datasets, names):
    profile = profiler.summary(df) # automatically prints some statistics and returns object containing stats

display(profile)

Total records across all profiling_datasets: 3,012
products_1:
  Rows: 812
  Columns: 27
  Total nulls: 9,142
  Null percentage: 41.7%
  Null counts per column:
    brand: 27 (3.3%)
    price: 40 (4.9%)
    priceCurrency: 43 (5.3%)
    model: 64 (7.9%)
    model_number: 478 (58.9%)
    chipset_name: 581 (71.6%)
    vram_gb: 583 (71.8%)
    storage_gb: 241 (29.7%)
    read_speed_mb_s: 661 (81.4%)
    write_speed_mb_s: 690 (85.0%)
    bus_type: 184 (22.7%)
    interface_type: 376 (46.3%)
    width_mm: 767 (94.5%)
    length_mm: 767 (94.5%)
    height_mm: 766 (94.3%)
    weight_g: 786 (96.8%)
    storage_connection_type: 399 (49.1%)
    memory_type: 602 (74.1%)
    color: 692 (85.2%)
    form_factor: 395 (48.6%)

products_2:
  Rows: 812
  Columns: 27
  Total nulls: 9,269
  Null percentage: 42.3%
  Null counts per column:
    brand: 41 (5.0%)
    price: 58 (7.1%)
    priceCurrency: 58 (7.1%)
    model: 70 (8.6%)
    model_number: 482 (59.4%)
    chipset_name: 581 (71.6%)
    vram_gb: 586 (

{'rows': 626,
 'columns': 27,
 'nulls_total': 7166,
 'nulls_per_column': {'id': 0,
  'brand': 34,
  'title': 0,
  'description': 0,
  'price': 43,
  'priceCurrency': 44,
  'cluster_id': 0,
  'url': 0,
  'title_description': 0,
  'model': 49,
  'model_number': 376,
  'product_type': 1,
  'chipset_name': 443,
  'vram_gb': 449,
  'storage_gb': 196,
  'read_speed_mb_s': 520,
  'write_speed_mb_s': 538,
  'bus_type': 139,
  'interface_type': 293,
  'width_mm': 599,
  'length_mm': 602,
  'height_mm': 591,
  'weight_g': 605,
  'storage_connection_type': 306,
  'memory_type': 464,
  'color': 574,
  'form_factor': 300},
 'dtypes': {'id': 'int64',
  'brand': 'object',
  'title': 'object',
  'description': 'object',
  'price': 'float64',
  'priceCurrency': 'object',
  'cluster_id': 'int64',
  'url': 'object',
  'title_description': 'object',
  'model': 'object',
  'model_number': 'object',
  'product_type': 'object',
  'chipset_name': 'object',
  'vram_gb': 'float64',
  'storage_gb': 'float64',
  

In [4]:
coverage = profiler.analyze_coverage(
    datasets=profiling_datasets,
    include_samples=True,
    sample_count=3  # Show 3 sample values per attribute
)

print("📊 Attribute coverage across profiling_datasets:")
display(coverage)

# Identify attributes suitable for entity matching
print("\n🔗 Attributes suitable for entity matching:")
matching_attrs = coverage[coverage['datasets_with_attribute'] >= 2]['attribute'].tolist()
print(f"Attributes available in 2+ datasets: {matching_attrs}")

📊 Attribute coverage across profiling_datasets:


,attribute,products_1_count,products_1_pct,products_1_coverage,products_1_samples,products_2_count,products_2_pct,products_2_coverage,products_2_samples,products_3_count,products_3_pct,products_3_coverage,products_3_samples,products_4_count,products_4_pct,products_4_coverage,products_4_samples,avg_coverage,max_coverage,datasets_with_attribute
0,brand,785/812,96.7%,0.966749,"['Gigabyte', 'Western Digital', 'Corsair']",771/812,95.0%,0.949507,"['Gigabyte', 'Western Digital', 'Corsair']",721/762,94.6%,0.946194,"['Gigabyte', 'Western Digital', 'Corsair']",592/626,94.6%,0.945687,"['Gigabyte', 'Western Digital', 'Corsair']",0.952034,0.966749,4
1,bus_type,628/812,77.3%,0.773399,"['SATA', 'PCI Express x4', 'SATA']",598/812,73.6%,0.736453,"['SATA', 'PCI Express x4', 'SATA']",573/762,75.2%,0.751969,"['SATA', 'SATA', 'PCI Express x16']",487/626,77.8%,0.777955,"['SATA', 'SATA', 'PCI Express x16']",0.759944,0.777955,4
2,chipset_name,231/812,28.4%,0.284483,"['GeForce RTX 3080', 'GeForce GTX 1650', 'GeFo...",231/812,28.4%,0.284483,"['GeForce RTX 3080', 'GeForce GTX 1650', 'GeFo...",219/762,28.7%,0.287402,"['GeForce RTX 3080', 'GeForce GTX 1650', 'GeFo...",183/626,29.2%,0.292332,"['GeForce RTX 3080', 'GeForce GTX 1650', 'GeFo...",0.287175,0.292332,4
3,cluster_id,812/812,100.0%,1.000000,"[1002037, 1004942, 1007272]",812/812,100.0%,1.000000,"[1002037, 1004942, 1007272]",762/762,100.0%,1.000000,"[1002037, 1004942, 1007272]",626/626,100.0%,1.000000,"[1002037, 1004942, 1007272]",1.000000,1.000000,4
4,color,120/812,14.8%,0.147783,"['black and red', 'Green', 'Black']",112/812,13.8%,0.137931,"['Black', 'Silver', 'Cobalt Trim']",81/762,10.6%,0.106299,"['svart/blå', 'Black/White', 'black']",52/626,8.3%,0.083067,"['ROSE GOLD', 'Black', 'Alb']",0.118770,0.147783,4
5,description,812/812,100.0%,1.000000,"['CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR...",812/812,100.0%,1.000000,['Gigabyte NVIDIA GeForce RTX 3080 GAMING OC 1...,762/762,100.0%,1.000000,"['To Avail the offer Click Here', 'Western Dig...",626/626,100.0%,1.000000,['Gigabyte Video Card GV-N3080GAMING OC-10GD G...,1.000000,1.000000,4
6,form_factor,417/812,51.4%,0.513547,"['3.5-inch', 'M.2 2280', '3.5-inch']",410/812,50.5%,0.504926,"['3.5-inch', 'M.2 2280', '3.5-inch']",374/762,49.1%,0.490814,"['3.5-inch', '3.5-inch', 'M.2 2280']",326/626,52.1%,0.520767,"['3.5-inch', 'M.2 2280', '3.5-inch']",0.507513,0.520767,4
7,height_mm,46/812,5.7%,0.056650,"[122.0, 6.1, 40.6]",48/812,5.9%,0.059113,"[127.0, 10.5, 11.4]",37/762,4.9%,0.048556,"[130.9, 20.8, 7.0]",35/626,5.6%,0.055911,"[7.0, 15.0, 119.3]",0.055058,0.059113,4
8,id,812/812,100.0%,1.000000,"[12198483, 78378158, 80641070]",812/812,100.0%,1.000000,"[19126355, 42841911, 46775597]",762/762,100.0%,1.000000,"[46320085, 91583813, 86850217]",626/626,100.0%,1.000000,"[66956099, 5078198, 77571226]",1.000000,1.000000,4
9,interface_type,436/812,53.7%,0.536946,"['SATA III', 'NVMe', 'SATA III']",424/812,52.2%,0.522167,"['SATA III', 'NVMe', 'SATA III']",381/762,50.0%,0.500000,"['SATA III', 'NVMe', 'SATA III']",333/626,53.2%,0.531949,"['SATA III', 'SATA III', 'NVMe']",0.522766,0.536946,4



🔗 Attributes suitable for entity matching:
Attributes available in 2+ datasets: ['brand', 'bus_type', 'chipset_name', 'cluster_id', 'color', 'description', 'form_factor', 'height_mm', 'id', 'interface_type', 'length_mm', 'memory_type', 'model', 'model_number', 'price', 'priceCurrency', 'product_type', 'read_speed_mb_s', 'storage_connection_type', 'storage_gb', 'title', 'title_description', 'url', 'vram_gb', 'weight_g', 'width_mm', 'write_speed_mb_s']


In [5]:
# Generate detailed HTML profiles for each dataset
profile_dir = OUTPUT_DIR / "dataset-profiles"
profile_dir.mkdir(parents=True, exist_ok=True)

profile_paths = []

for df, name in zip(profiling_datasets, names): #will put the most upto date datasets here after normalization
    print(f"Profiling {name}...")
    
    profile_path = profiler.profile(df, str(profile_dir))
    profile_paths.append(profile_path)
    print(f"Profile saved: {profile_path}")

print(f"\n Generated {len(profile_paths)} detailed HTML reports")
print(f" Location: {profile_dir}")
print("\n Open these HTML files in your browser for interactive exploration:")
for path in profile_paths:
    print(f"  • {Path(path).name}")

Profiling products_1...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 43.27it/s]


Profile saved: C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\dataset-profiles\products_1_profile.html
Profiling products_2...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 51.03it/s]


Profile saved: C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\dataset-profiles\products_2_profile.html
Profiling products_3...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 33.36it/s]


Profile saved: C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\dataset-profiles\products_3_profile.html
Profiling products_4...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 47.59it/s]

Profile saved: C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\dataset-profiles\products_4_profile.html

 Generated 4 detailed HTML reports
 Location: C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\dataset-profiles

 Open these HTML files in your browser for interactive exploration:
  • products_1_profile.html
  • products_2_profile.html
  • products_3_profile.html
  • products_4_profile.html


## Part 2: Entity Matching

**setup logging**

In [6]:
# Set up logging
os.makedirs('output/logs', exist_ok=True)

logging.basicConfig(
    level=logging.INFO, # Alternatively, use logging.DEBUG for more verbosity
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler('output/logs/pydi.log'),  # Save to file
          logging.StreamHandler()                      # Display on console
      ],
    force=True
)

### Step 1: Blocking

In [7]:
out_dir_blocker =OUTPUT_DIR / "Blocking" / "standard_blocker_on_product_type"
out_dir_blocker.mkdir(parents=True, exist_ok=True)


# 3. Initialize the Star Schema Blockers
standard_blocker_p1_p2 = StandardBlocker(
    products_1_cleaned, products_2_cleaned,
    on=['product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p2_blocking",
    id_column='id'
)

standard_blocker_p1_p3 = StandardBlocker(
    products_1_cleaned, products_3_cleaned,
    on=['product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p3_blocking",
    id_column='id'
)

standard_blocker_p1_p4 = StandardBlocker(
    products_1_cleaned, products_4_cleaned,
    on=[ 'product_type'],
    batch_size=1000,
    output_dir=out_dir_blocker / "p1_p4_blocking",
    id_column='id'
)

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\Blocking\standard_blocker_on_product_type\p1_p2_blocking\debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 5 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardB

### Step 2: Evaluate Blocking Against Ground Truth

In [8]:
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt_p1_p2 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod2_val.csv", 
    name="test_p1_p2", header=0, names=['id1', 'id2', 'label','ishard'], add_index=False)

# 2. Evaluate Blocker
results_p1_p2 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p2,
    test_pairs=test_gt_p1_p2,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod2"
)

print("--- Evaluation: Product 1 to Product 2 ---")
display(results_p1_p2)



[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 3 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 10 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 11 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 17 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 19 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 23 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 26 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 29 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 30 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 36 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 39 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 42 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 45 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 48 true matches
[INFO ] root

--- Evaluation: Product 1 to Product 2 ---


{'pair_completeness': 1.0,
 'pair_quality': 0.0003358431287579221,
 'reduction_ratio': 0.7200095852847679,
 'total_candidates': 184610,
 'total_possible_pairs': 659344,
 'true_positives_found': 62,
 'total_true_pairs': 62,
 'batches_processed': 185,
 'evaluation_timestamp': '2026-04-22T20:42:22.532694',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\Blocking\\blocking_eval_prod1_prod2\\blocking_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\Blocking\\blocking_eval_prod1_prod2\\blocking_detailed_results.csv']}

In [9]:
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt_p1_p3 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod3_val.csv", 
    name="test_p1_p3", header=0, names=['id1', 'id2', 'label', 'ishard'], add_index=False)

# 2. Evaluate Blocker
results_p1_p3 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p3,
    test_pairs=test_gt_p1_p3,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod3"
)

print("--- Evaluation: Product 1 to Product 3 ---")
display(results_p1_p3)


[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 3 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 11 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 13 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 18 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 21 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 24 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 26 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 29 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 35 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 38 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 42 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 45 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 48 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 52 true matches
[INFO ] root

--- Evaluation: Product 1 to Product 3 ---


{'pair_completeness': 1.0,
 'pair_quality': 0.0003577796885008454,
 'reduction_ratio': 0.7199310215533403,
 'total_candidates': 173291,
 'total_possible_pairs': 618744,
 'true_positives_found': 62,
 'total_true_pairs': 62,
 'batches_processed': 174,
 'evaluation_timestamp': '2026-04-22T20:42:34.862649',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\Blocking\\blocking_eval_prod1_prod3\\blocking_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\Blocking\\blocking_eval_prod1_prod3\\blocking_detailed_results.csv']}

In [10]:
# Showcase EntityMatchingEvaluator.evaluate_blocking utility

# Load test set with proper column names
test_gt_p1_p4 = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod4_val.csv", 
    name="test_p1_p4", header=0, names=['id1', 'id2', 'label','ishard'], add_index=False)

# 2. Evaluate Blocker
results_p1_p4 = EntityMatchingEvaluator.evaluate_blocking_batched(
    blocker=standard_blocker_p1_p4,
    test_pairs=test_gt_p1_p4,
    out_dir=OUTPUT_DIR / "Blocking" / "blocking_eval_prod1_prod4"
)

print("--- Evaluation: Product 1 to Product 4 ---")
display(results_p1_p4)


[INFO ] root - Starting batched blocking evaluation...
[INFO ] root - Processed 10 batches, 10000 pairs, 4 true matches
[INFO ] root - Processed 20 batches, 20000 pairs, 11 true matches
[INFO ] root - Processed 30 batches, 30000 pairs, 16 true matches
[INFO ] root - Processed 40 batches, 40000 pairs, 19 true matches
[INFO ] root - Processed 50 batches, 50000 pairs, 24 true matches
[INFO ] root - Processed 60 batches, 60000 pairs, 28 true matches
[INFO ] root - Processed 70 batches, 70000 pairs, 30 true matches
[INFO ] root - Processed 80 batches, 80000 pairs, 38 true matches
[INFO ] root - Processed 90 batches, 90000 pairs, 40 true matches
[INFO ] root - Processed 100 batches, 100000 pairs, 45 true matches
[INFO ] root - Processed 110 batches, 110000 pairs, 48 true matches
[INFO ] root - Processed 120 batches, 120000 pairs, 53 true matches
[INFO ] root - Processed 130 batches, 130000 pairs, 56 true matches
[INFO ] root - Processed 140 batches, 140000 pairs, 58 true matches
[INFO ] root

--- Evaluation: Product 1 to Product 4 ---


{'pair_completeness': 1.0,
 'pair_quality': 0.00043072904364257824,
 'reduction_ratio': 0.71682352570862,
 'total_candidates': 143942,
 'total_possible_pairs': 508312,
 'true_positives_found': 62,
 'total_true_pairs': 62,
 'batches_processed': 144,
 'evaluation_timestamp': '2026-04-22T20:42:46.672816',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\Blocking\\blocking_eval_prod1_prod4\\blocking_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\Blocking\\blocking_eval_prod1_prod4\\blocking_detailed_results.csv']}

### Step 3: Entity Matching with Comparators

In [11]:
# Custom hardware normalization (similar to music/companies)
# This removes noise like "GB", "TB", and punctuation to focus on model codes
def normalize_hardware_text(s: str) -> str: 
    if s is None:
        return ""
    # Remove standard units and punctuation to isolate model numbers (e.g., 980, SN850)
    s = re.sub(r"(?i)\b(gb|tb|ssd|nvme|internal|hhd|sata)\b", "", s)
    return re.sub(r"[^\w\s]|_", "", s).lower().strip()


comparators = [
    # 1. Title (Tokens) - Handles word shuffling
    StringComparator(
        column='title', 
        similarity_function='sorensen_dice', #got better results than jaccard
        tokenization='word',
        preprocess=normalize_hardware_text
    ),
    
    # 2. Brand - Switched to Jaccard since they are normalized
    StringComparator(
        column='brand',
        similarity_function='jaccard',
        tokenization='word',
        preprocess=str.lower
    ),
    
    # 3. Product Type - Switched to Jaccard for strict category matching
    StringComparator(
        column='product_type',
        similarity_function='jaccard',
        tokenization='word',
        preprocess=str.lower
    ),
    
    # 4. Storage GB - Essential to distinguish 500GB from 1000GB variants
    NumericComparator(
        column='storage_gb',
        method='relative_difference',
        max_difference=0.1 
    )
]

Next, we setup the matcher and run the matching with our chosen best blocking method:

### Step 4: Rule Based Matcher

In [12]:
matcher = RuleBasedMatcher()

# New Weights distribution (must sum to 1.0):
# [Title_sorensen_dice, Brand_Jaccard, product_Type_Jaccard, Storage_Numeric]
# current_weights = [0.35, 0.30, 0.20, 0.15] #good balance, decent F1
# current_weights = [0.35, 0.30, 0.25, 0.10] #good for all, slightly better F1 than above (2nd best)
# current_weights = [0.30, 0.25, 0.30, 0.15] #slightly worse
current_weights = [0.30, 0.30, 0.30, 0.10] #best so far
# current_weights = [0.45, 0.15, 0.10, 0.30] #perfect precision but sinks in recall and f1
# current_weights = [0.35, 0.30, 0.30, 0.05] #worse than above
# current_weights = [0.50, 0.15, 0.30, 0.05] #massive drop in recall and F1, even if precision is perfect, so not good
# current_weights = [0.40, 0.20, 0.30, 0.10] #2nd best

current_threshold = 0.70 

# Evaluation for P1 to P2
correspondences_p1_p2 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_2_cleaned,
    candidates=standard_blocker_p1_p2, 
    comparators=comparators, 
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

# Evaluation for P1 to P3
correspondences_p1_p3 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_3_cleaned,
    candidates=standard_blocker_p1_p3,
    comparators=comparators,
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

# Evaluation for P1 to P4
correspondences_p1_p4 = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_4_cleaned,
    candidates=standard_blocker_p1_p4,
    comparators=comparators,
    weights=current_weights,
    threshold=current_threshold,
    id_column='id'
)

[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 812 x 812 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 812 x 812 elements after 0:00:0.162; 184610 blocked pairs (reduction ratio: 0.7200095852847679)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:48.772; found 13250 correspondences.
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Blocking 812 x 762 elements
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Matching 812 x 762 elements after 0:00:0.114; 173291 blocked pairs (reduction ratio: 0.7199310215533403)
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Entity Matching finished after 0:00:45.550; found 11287 correspondences.
[INFO ] PyDI.entitymatching.rule_based.RuleBasedMatcher - Starting Entity 

### Step 5: Evaluate Matching Against Ground Truth

In [13]:
# Evaluate P1 to P2
results_p1_p2 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p2,
    test_pairs=test_gt_p1_p2,
 
)

# Evaluate P1 to P3
results_p1_p3 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p3,
    test_pairs=test_gt_p1_p3,

)

# Evaluate P1 to P4
results_p1_p4 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p4,
    test_pairs=test_gt_p1_p4,
 
)

[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  56
[INFO ] root -   True Negatives:  71
[INFO ] root -   False Positives: 19
[INFO ] root -   False Negatives: 6
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.836
[INFO ] root -   Precision: 0.747
[INFO ] root -   Recall:    0.903
[INFO ] root -   F1-Score:  0.818
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  55
[INFO ] root -   True Negatives:  43
[INFO ] root -   False Positives: 5
[INFO ] root -   False Negatives: 7
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.891
[INFO ] root -   Precision: 0.917
[INFO ] root -   Recall:    0.887
[INFO ] root -   F1-Score:  0.902
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  58
[INFO ] root -   True Negatives:  50
[INFO ] root -   False Positives: 4
[INFO ] root -   False Negatives: 4
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.931
[INFO ] root -   Precision: 0.935
[INFO ] root -

In [14]:
print("--- P1 to P2 Matching Results ---")
display(results_p1_p2)

print("--- P1 to P3 Matching Results ---")
display(results_p1_p3)

print("--- P1 to P4 Matching Results ---")
display(results_p1_p4)

--- P1 to P2 Matching Results ---


{'precision': 0.7466666666666667,
 'recall': 0.9032258064516129,
 'f1': 0.8175182481751825,
 'accuracy': 0.8355263157894737,
 'true_positives': 56,
 'false_positives': 19,
 'false_negatives': 6,
 'true_negatives': 71,
 'threshold_used': 0.0,
 'total_correspondences': 13250,
 'filtered_correspondences': 13250,
 'evaluation_timestamp': '2026-04-22T20:45:17.922559'}

--- P1 to P3 Matching Results ---


{'precision': 0.9166666666666666,
 'recall': 0.8870967741935484,
 'f1': 0.9016393442622951,
 'accuracy': 0.8909090909090909,
 'true_positives': 55,
 'false_positives': 5,
 'false_negatives': 7,
 'true_negatives': 43,
 'threshold_used': 0.0,
 'total_correspondences': 11287,
 'filtered_correspondences': 11287,
 'evaluation_timestamp': '2026-04-22T20:45:18.625474'}

--- P1 to P4 Matching Results ---


{'precision': 0.9354838709677419,
 'recall': 0.9354838709677419,
 'f1': 0.9354838709677419,
 'accuracy': 0.9310344827586207,
 'true_positives': 58,
 'false_positives': 4,
 'false_negatives': 4,
 'true_negatives': 50,
 'threshold_used': 0.0,
 'total_correspondences': 8991,
 'filtered_correspondences': 8991,
 'evaluation_timestamp': '2026-04-22T20:45:19.199507'}

In [15]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p2,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p2_cluster_distribution")
)

print(f"\n📊 Cluster Size Distribution Results for p1_p2:")
display(cluster_distribution)

# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p2_cluster_distribution" / "p1_p2_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p2,
    out_path=cluster_details_path
)


Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 55 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	13	|	23.64%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	4	|	7.27%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	5	|	9.09%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	1	|	1.82%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	3	|	5.45%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	1	|	1.82%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	3	|	5.45%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	1	|	1.82%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	2	|	3.64%
[INFO ] PyDI.entitymatching.evaluation - 		14	|	2	|	3.64%
[INFO ] PyDI.entitymatching.evaluation - 		17	|	1	|	1.82%
[INFO ] PyDI.entitymatching.evaluation - 		20	|	2	|	3.64%
[INFO ] PyDI.entitymatching.evaluation - 		29	|	1	|	1


📊 Cluster Size Distribution Results for p1_p2:


,cluster_size,frequency,percentage
0,2,13,23.636364
1,3,4,7.272727
2,4,5,9.090909
3,5,1,1.818182
4,6,3,5.454545
5,7,1,1.818182
6,8,3,5.454545
7,10,1,1.818182
8,12,2,3.636364
9,14,2,3.636364


[INFO ] root - Cluster details written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p2_cluster_distribution\p1_p2_detailed_cluster_info.json
[INFO ] root - Exported 55 clusters with detailed record information


In [16]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p3,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p3_cluster_distribution")
)

print(f"\n📊 Cluster Size Distribution Results for p1_p3:")
display(cluster_distribution)
# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p3_cluster_distribution" / "p1_p3_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p3,
    out_path=cluster_details_path
)


Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 56 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	16	|	28.57%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	3	|	5.36%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	4	|	7.14%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	2	|	3.57%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	1	|	1.79%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	2	|	3.57%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	4	|	7.14%
[INFO ] PyDI.entitymatching.evaluation - 		10	|	2	|	3.57%
[INFO ] PyDI.entitymatching.evaluation - 		14	|	1	|	1.79%
[INFO ] PyDI.entitymatching.evaluation - 		15	|	2	|	3.57%
[INFO ] PyDI.entitymatching.evaluation - 		19	|	1	|	1.79%
[INFO ] PyDI.entitymatching.evaluation - 		20	|	1	|	1.79%
[INFO ] PyDI.entitymatching.evaluation - 		28	|	1	|	1


📊 Cluster Size Distribution Results for p1_p3:


,cluster_size,frequency,percentage
0,2,16,28.571429
1,3,3,5.357143
2,4,4,7.142857
3,5,2,3.571429
4,6,1,1.785714
5,7,2,3.571429
6,8,4,7.142857
7,10,2,3.571429
8,14,1,1.785714
9,15,2,3.571429


[INFO ] root - Cluster details written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p3_cluster_distribution\p1_p3_detailed_cluster_info.json
[INFO ] root - Exported 56 clusters with detailed record information


In [17]:
# This prevents the logger from crashing when it can't print a special character
logging.raiseExceptions = False

print("Analyzing cluster size distribution in our entity matching results...")

# Create cluster size distribution from our matches
cluster_distribution = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p4,
    out_dir=str(OUTPUT_DIR / "cluster_analysis"/ "p1_p4_cluster_distribution")
)

print(f"\n📊 Cluster Size Distribution Results for p1_p4:")
display(cluster_distribution)

# Write out detailed cluster information with all entity records for debugging purposes

# Use the matches we found earlier to demonstrate cluster details
cluster_details_path = OUTPUT_DIR / "cluster_analysis" / "p1_p4_cluster_distribution" / "p1_p4_detailed_cluster_info.json"

# Call write_cluster_details with our entity matches
output_path = EntityMatchingEvaluator.write_cluster_details(
    correspondences=correspondences_p1_p4,
    out_path=cluster_details_path
)

Analyzing cluster size distribution in our entity matching results...


[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 43 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	6	|	13.95%
[INFO ] PyDI.entitymatching.evaluation - 		3	|	2	|	4.65%
[INFO ] PyDI.entitymatching.evaluation - 		4	|	4	|	9.30%
[INFO ] PyDI.entitymatching.evaluation - 		5	|	1	|	2.33%
[INFO ] PyDI.entitymatching.evaluation - 		6	|	2	|	4.65%
[INFO ] PyDI.entitymatching.evaluation - 		7	|	3	|	6.98%
[INFO ] PyDI.entitymatching.evaluation - 		8	|	2	|	4.65%
[INFO ] PyDI.entitymatching.evaluation - 		12	|	2	|	4.65%
[INFO ] PyDI.entitymatching.evaluation - 		13	|	1	|	2.33%
[INFO ] PyDI.entitymatching.evaluation - 		14	|	1	|	2.33%
[INFO ] PyDI.entitymatching.evaluation - 		18	|	1	|	2.33%
[INFO ] PyDI.entitymatching.evaluation - 		20	|	1	|	2.33%
[INFO ] PyDI.entitymatching.evaluation - 		24	|	1	|	2.


📊 Cluster Size Distribution Results for p1_p4:


,cluster_size,frequency,percentage
0,2,6,13.953488
1,3,2,4.651163
2,4,4,9.302326
3,5,1,2.325581
4,6,2,4.651163
5,7,3,6.976744
6,8,2,4.651163
7,12,2,4.651163
8,13,1,2.325581
9,14,1,2.325581


[INFO ] root - Cluster details written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p4_cluster_distribution\p1_p4_detailed_cluster_info.json
[INFO ] root - Exported 43 clusters with detailed record information


Additionally, PyDI offers 6 different post-clustering methods to "clean" clusters after entity matching. For example, if we want to enforce that each record in a dataset can only have exactly one correspondence in the other dataset, we can apply a greedy one-to-one matching, maximum bipartite matching or stable marriage matching.

### Step 6: GreedyOneToOneMatchingAlgorithm

In [18]:
# 1. Initialize the Refinement Clusterer
# This ensures that one item in P1 doesn't match multiple items in P2
refiner = GreedyOneToOneMatchingAlgorithm()

print("Refining P1-P2 results to 1:1 matches...")
correspondences_p1_p2_refined = refiner.cluster(correspondences_p1_p2)

# 2. Re-Evaluate the REFINED Results
# You should see your Precision go UP, but Recall might go down slightly
results_p1_p2_refined = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p2_refined,
    test_pairs=test_gt_p1_p2,
    out_dir=OUTPUT_DIR / "debug_results_entity_matching" / "p1_p2_refined"
)

print("\n✨ Refined Performance Metrics (P1 to P2):")
display(results_p1_p2_refined)

print("Analyzing REFINED cluster size distribution...")

# Create distribution for the 1:1 results
cluster_dist_refined = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p2_refined,
    out_dir=str(OUTPUT_DIR / "cluster_analysis" / "p1_p2_greedy_1_to_1_distribution")
)

display(cluster_dist_refined)


[INFO ] root - Filtered correspondences: 13250 -> 13250 (threshold=0.0)


Refining P1-P2 results to 1:1 matches...


[INFO ] root - Greedy matching: 13250 -> 720 correspondences (1440 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 13250 -> 720 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 1526 -> 1440 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  27
[INFO ] root -   True Negatives:  88
[INFO ] root -   False Positives: 2
[INFO ] root -   False Negatives: 35
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.757
[INFO ] root -   Precision: 0.931
[INFO ] root -   Recall:    0.435
[INFO ] root -   F1-Score:  0.593



✨ Refined Performance Metrics (P1 to P2):


{'precision': 0.9310344827586207,
 'recall': 0.43548387096774194,
 'f1': 0.5934065934065935,
 'accuracy': 0.756578947368421,
 'true_positives': 27,
 'false_positives': 2,
 'false_negatives': 35,
 'true_negatives': 88,
 'threshold_used': 0.0,
 'total_correspondences': 720,
 'filtered_correspondences': 720,
 'evaluation_timestamp': '2026-04-22T20:45:24.511917',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p2_refined\\matching_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p2_refined\\matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 720 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	720	|	100.00%
[INFO ] root - Cluster size distribution written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p2_greedy_1_to_1_distribution\cluster_size_distribution.csv


Analyzing REFINED cluster size distribution...


,cluster_size,frequency,percentage
0,2,720,100.0


In [19]:
# 1. Initialize the Refinement Clusterer
# This ensures that one item in P1 doesn't match multiple items in P3
refiner = GreedyOneToOneMatchingAlgorithm()

print("Refining P1-P3 results to 1:1 matches...")
correspondences_p1_p3_refined = refiner.cluster(correspondences_p1_p3)
# 2. Re-Evaluate the REFINED Results
# You should see your Precision go UP, but Recall might go down slightly
results_p1_p3_refined = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p3_refined,
    test_pairs=test_gt_p1_p3,
    out_dir=OUTPUT_DIR / "debug_results_entity_matching" / "p1_p3_refined"
)

print("\n✨ Refined Performance Metrics (P1 to P3):")
display(results_p1_p3_refined)

print("Analyzing REFINED cluster size distribution...")

# Create distribution for the 1:1 results
cluster_dist_refined = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p3_refined,
    out_dir=str(OUTPUT_DIR / "cluster_analysis" / "p1_p3_greedy_1_to_1_distribution")
)

display(cluster_dist_refined)


[INFO ] root - Filtered correspondences: 11287 -> 11287 (threshold=0.0)


Refining P1-P3 results to 1:1 matches...


[INFO ] root - Greedy matching: 11287 -> 688 correspondences (1376 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 11287 -> 688 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 1483 -> 1376 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  36
[INFO ] root -   True Negatives:  47
[INFO ] root -   False Positives: 1
[INFO ] root -   False Negatives: 26
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.755
[INFO ] root -   Precision: 0.973
[INFO ] root -   Recall:    0.581
[INFO ] root -   F1-Score:  0.727



✨ Refined Performance Metrics (P1 to P3):


{'precision': 0.972972972972973,
 'recall': 0.5806451612903226,
 'f1': 0.7272727272727273,
 'accuracy': 0.7545454545454545,
 'true_positives': 36,
 'false_positives': 1,
 'false_negatives': 26,
 'true_negatives': 47,
 'threshold_used': 0.0,
 'total_correspondences': 688,
 'filtered_correspondences': 688,
 'evaluation_timestamp': '2026-04-22T20:45:25.200764',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p3_refined\\matching_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p3_refined\\matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 688 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	688	|	100.00%
[INFO ] root - Cluster size distribution written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p3_greedy_1_to_1_distribution\cluster_size_distribution.csv


Analyzing REFINED cluster size distribution...


,cluster_size,frequency,percentage
0,2,688,100.0


In [20]:
# 1. Initialize the Refinement Clusterer
# This ensures that one item in P1 doesn't match multiple items in P4
refiner = GreedyOneToOneMatchingAlgorithm()

print("Refining P1-P4 results to 1:1 matches...")
correspondences_p1_p4_refined = refiner.cluster(correspondences_p1_p4)
# 2. Re-Evaluate the REFINED Results
# You should see your Precision go UP, but Recall might go down slightly
results_p1_p4_refined = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p4_refined,
    test_pairs=test_gt_p1_p4,
    out_dir=OUTPUT_DIR / "debug_results_entity_matching" / "p1_p4_refined"
)

print("\n✨ Refined Performance Metrics (P1 to P4):")
display(results_p1_p4_refined)

print("Analyzing REFINED cluster size distribution...")

# Create distribution for the 1:1 results
cluster_dist_refined = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p4_refined,
    out_dir=str(OUTPUT_DIR / "cluster_analysis" / "p1_p4_greedy_1_to_1_distribution")
)

display(cluster_dist_refined)


[INFO ] root - Filtered correspondences: 8991 -> 8991 (threshold=0.0)


Refining P1-P4 results to 1:1 matches...


[INFO ] root - Greedy matching: 8991 -> 568 correspondences (1136 entities matched)
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 8991 -> 568 correspondences
[INFO ] root - GreedyOneToOneMatchingAlgorithm: 1332 -> 1136 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  33
[INFO ] root -   True Negatives:  54
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 29
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.750
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.532
[INFO ] root -   F1-Score:  0.695



✨ Refined Performance Metrics (P1 to P4):


{'precision': 1.0,
 'recall': 0.532258064516129,
 'f1': 0.6947368421052631,
 'accuracy': 0.75,
 'true_positives': 33,
 'false_positives': 0,
 'false_negatives': 29,
 'true_negatives': 54,
 'threshold_used': 0.0,
 'total_correspondences': 568,
 'filtered_correspondences': 568,
 'evaluation_timestamp': '2026-04-22T20:45:25.791632',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p4_refined\\matching_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p4_refined\\matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 568 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	568	|	100.00%
[INFO ] root - Cluster size distribution written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p4_greedy_1_to_1_distribution\cluster_size_distribution.csv


Analyzing REFINED cluster size distribution...


,cluster_size,frequency,percentage
0,2,568,100.0


### Step 7: MaximumBipartiteMatching

In [21]:
# 1. Initialize the MBM Clusterer
mbm_clusterer = MaximumBipartiteMatching()

print("Refining P1-P2 results using Maximum Bipartite Matching...")
# Use your ORIGINAL (unrefined) correspondences here to see if MBM is better than Greedy
correspondences_p1_p2_mbm = mbm_clusterer.cluster(correspondences_p1_p2)

# 2. Evaluate the MBM Results
results_p1_p2_mbm = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p2_mbm,
    test_pairs=test_gt_p1_p2,
    out_dir=OUTPUT_DIR / "debug_results_entity_matching" / "p1_p2_mbm"
)

print("\n🏆 MBM Performance Metrics (P1 to P2):")
display(results_p1_p2_mbm)

# 3. Check Cluster Size (Should still be 100% size 2)
mbm_dist = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p2_mbm,
    out_dir=str(OUTPUT_DIR / "cluster_analysis" / "p1_p2_mbm_distribution")
)
display(mbm_dist)

[INFO ] root - Filtered correspondences: 13250 -> 13250 (threshold=0.0)


Refining P1-P2 results using Maximum Bipartite Matching...


[INFO ] root - Maximum bipartite matching: 13250 -> 745 
[INFO ] root - MaximumBipartiteMatching: 13250 -> 745 correspondences
[INFO ] root - MaximumBipartiteMatching: 1526 -> 1490 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  5
[INFO ] root -   True Negatives:  87
[INFO ] root -   False Positives: 3
[INFO ] root -   False Negatives: 57
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.605
[INFO ] root -   Precision: 0.625
[INFO ] root -   Recall:    0.081
[INFO ] root -   F1-Score:  0.143



🏆 MBM Performance Metrics (P1 to P2):


{'precision': 0.625,
 'recall': 0.08064516129032258,
 'f1': 0.14285714285714285,
 'accuracy': 0.6052631578947368,
 'true_positives': 5,
 'false_positives': 3,
 'false_negatives': 57,
 'true_negatives': 87,
 'threshold_used': 0.0,
 'total_correspondences': 745,
 'filtered_correspondences': 745,
 'evaluation_timestamp': '2026-04-22T20:45:27.389847',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p2_mbm\\matching_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p2_mbm\\matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 745 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	745	|	100.00%
[INFO ] root - Cluster size distribution written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p2_mbm_distribution\cluster_size_distribution.csv


,cluster_size,frequency,percentage
0,2,745,100.0


In [22]:
# 1. Initialize the MBM Clusterer
mbm_clusterer = MaximumBipartiteMatching()

print("Refining P1-P3 results using Maximum Bipartite Matching...")
# Use your ORIGINAL (unrefined) correspondences here to see if MBM is better than Greedy
correspondences_p1_p3_mbm = mbm_clusterer.cluster(correspondences_p1_p3)

# 2. Evaluate the MBM Results
results_p1_p3_mbm = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p3_mbm,
    test_pairs=test_gt_p1_p3,
    out_dir=OUTPUT_DIR / "debug_results_entity_matching" / "p1_p3_mbm"
)

print("\n🏆 MBM Performance Metrics (P1 to P3):")
display(results_p1_p3_mbm)

# 3. Check Cluster Size (Should still be 100% size 2)
mbm_dist = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p3_mbm,
    out_dir=str(OUTPUT_DIR / "cluster_analysis" / "p1_p3_mbm_distribution")
)
display(mbm_dist)

[INFO ] root - Filtered correspondences: 11287 -> 11287 (threshold=0.0)


Refining P1-P3 results using Maximum Bipartite Matching...


[INFO ] root - Maximum bipartite matching: 11287 -> 707 
[INFO ] root - MaximumBipartiteMatching: 11287 -> 707 correspondences
[INFO ] root - MaximumBipartiteMatching: 1483 -> 1414 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  8
[INFO ] root -   True Negatives:  48
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 54
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.509
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.129
[INFO ] root -   F1-Score:  0.229



🏆 MBM Performance Metrics (P1 to P3):


{'precision': 1.0,
 'recall': 0.12903225806451613,
 'f1': 0.2285714285714286,
 'accuracy': 0.509090909090909,
 'true_positives': 8,
 'false_positives': 0,
 'false_negatives': 54,
 'true_negatives': 48,
 'threshold_used': 0.0,
 'total_correspondences': 707,
 'filtered_correspondences': 707,
 'evaluation_timestamp': '2026-04-22T20:45:28.788559',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p3_mbm\\matching_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p3_mbm\\matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 707 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	707	|	100.00%
[INFO ] root - Cluster size distribution written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p3_mbm_distribution\cluster_size_distribution.csv


,cluster_size,frequency,percentage
0,2,707,100.0


In [23]:
# 1. Initialize the MBM Clusterer
mbm_clusterer = MaximumBipartiteMatching()

print("Refining P1-P4 results using Maximum Bipartite Matching...")
# Use your ORIGINAL (unrefined) correspondences here to see if MBM is better than Greedy
correspondences_p1_p4_mbm = mbm_clusterer.cluster(correspondences_p1_p4)

# 2. Evaluate the MBM Results
results_p1_p4_mbm = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_p1_p4_mbm,
    test_pairs=test_gt_p1_p4,
    out_dir=OUTPUT_DIR / "debug_results_entity_matching" / "p1_p4_mbm"
)

print("\n🏆 MBM Performance Metrics (P1 to P4):")
display(results_p1_p4_mbm)

# 3. Check Cluster Size (Should still be 100% size 2)
mbm_dist = EntityMatchingEvaluator.create_cluster_size_distribution(
    correspondences=correspondences_p1_p4_mbm,
    out_dir=str(OUTPUT_DIR / "cluster_analysis" / "p1_p4_mbm_distribution")
)
display(mbm_dist)

[INFO ] root - Filtered correspondences: 8991 -> 8991 (threshold=0.0)


Refining P1-P4 results using Maximum Bipartite Matching...


[INFO ] root - Maximum bipartite matching: 8991 -> 580 
[INFO ] root - MaximumBipartiteMatching: 8991 -> 580 correspondences
[INFO ] root - MaximumBipartiteMatching: 1332 -> 1160 entities
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  9
[INFO ] root -   True Negatives:  54
[INFO ] root -   False Positives: 0
[INFO ] root -   False Negatives: 53
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.543
[INFO ] root -   Precision: 1.000
[INFO ] root -   Recall:    0.145
[INFO ] root -   F1-Score:  0.254



🏆 MBM Performance Metrics (P1 to P4):


{'precision': 1.0,
 'recall': 0.14516129032258066,
 'f1': 0.2535211267605634,
 'accuracy': 0.5431034482758621,
 'true_positives': 9,
 'false_positives': 0,
 'false_negatives': 53,
 'true_negatives': 54,
 'threshold_used': 0.0,
 'total_correspondences': 580,
 'filtered_correspondences': 580,
 'evaluation_timestamp': '2026-04-22T20:45:30.156951',
 'output_files': ['C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p4_mbm\\matching_evaluation_summary.json',
  'C:\\Users\\hussa\\OneDrive\\Desktop\\Mannheim\\Semester 1\\IE 500 Data Mining\\Excercises\\00_PythonIntro\\hiwi_data_integration\\output\\debug_results_entity_matching\\p1_p4_mbm\\matching_detailed_results.csv']}

[INFO ] PyDI.entitymatching.evaluation - Cluster Size Distribution of 580 clusters:
[INFO ] PyDI.entitymatching.evaluation - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.entitymatching.evaluation - 	──────────────────────────────────────────────────
[INFO ] PyDI.entitymatching.evaluation - 		2	|	580	|	100.00%
[INFO ] root - Cluster size distribution written to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\cluster_analysis\p1_p4_mbm_distribution\cluster_size_distribution.csv


,cluster_size,frequency,percentage
0,2,580,100.0


### Step 8: ML-Based Matcher

In [24]:
# Load the specific splits for P1 -> P2
star_train = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod2_train.csv",
    add_index=False
)

star_test = load_csv(
    OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod2_test.csv",
    add_index=False
)
print("star_train columns:", star_train.columns.tolist())
print("star_train id1 dtype:", star_train['id1'].dtype)
print("star_train head:")
print(star_train.head(3))

print("\nproducts_1_cleaned id dtype:", products_1_cleaned['id'].dtype)
print("products_2_cleaned id dtype:", products_2_cleaned['id'].dtype)

# Check actual overlap
valid = set(products_1_cleaned['id']).union(set(products_2_cleaned['id']))
in_train = set(star_train['id1']).union(set(star_train['id2']))
phantom = in_train - valid
print(f"\nPhantom IDs in star_train: {len(phantom)}")
print(f"Sample phantom: {list(phantom)[:5]}")


# 2. Extract Features for Training
extractor = FeatureExtractor(comparators)
train_features = extractor.create_features(
    df_left=products_1_cleaned, 
    df_right=products_2_cleaned, 
    pairs=star_train, 
    id_column='id',
    labels=star_train['label']
)

# 3. Train the Classifier
X_train = train_features.drop(['label', 'id1', 'id2'], axis=1)
y_train = train_features['label']

# RandomForest is excellent for handling the mix of Jaccard and Levenshtein scores
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

# 4. Use MLBasedMatcher to find matches in the candidates
matcher = MLBasedMatcher(extractor)
correspondences_ml = matcher.match(
    df_left=products_1_cleaned,
    df_right=products_2_cleaned,
    candidates=standard_blocker_p1_p2, 
    id_column='id',
    trained_classifier=clf,
    threshold=0.45, #was 0.5
    use_probabilities=True
)

# 5. Evaluate against VAL
results_ml_val = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_ml,
    test_pairs=star_test
)

print("--- ML Results for P1 to P2 (Test Set) ---")
display(results_ml_val)



star_train columns: ['id1', 'id2', 'label', 'is_hard']
star_train id1 dtype: int64
star_train head:
        id1       id2  label  is_hard
0  12462933  61382599      1    False
1  12612826  42113643      0     True
2  50232345  69662078      1    False

products_1_cleaned id dtype: int64
products_2_cleaned id dtype: int64

Phantom IDs in star_train: 0
Sample phantom: []


[INFO ] root - Label distribution: 291 positive, 424 negative
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 812 x 812 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 812 x 812 elements after 0:00:0.199; 184610 blocked pairs (reduction ratio: 0.7200095852847679)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:63.741; found 26383 correspondences.
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  99
[INFO ] root -   True Negatives:  207
[INFO ] root -   False Positives: 44
[INFO ] root -   False Negatives: 34
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.797
[INFO ] root -   Precision: 0.692
[INFO ] root -   Recall:    0.744
[INFO ] root -   F1-Score:  0.717


--- ML Results for P1 to P2 (Test Set) ---


{'precision': 0.6923076923076923,
 'recall': 0.7443609022556391,
 'f1': 0.717391304347826,
 'accuracy': 0.796875,
 'true_positives': 99,
 'false_positives': 44,
 'false_negatives': 34,
 'true_negatives': 207,
 'threshold_used': 0.0,
 'total_correspondences': 26383,
 'filtered_correspondences': 26383,
 'evaluation_timestamp': '2026-04-22T20:46:36.992892'}

In [25]:
# P1 -> P3
star_train_p3 = load_csv(OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod3_train.csv", add_index=False)
star_val_p3   = load_csv(OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod3_val.csv",   add_index=False)

train_features_p3 = extractor.create_features(
    df_left=products_1_cleaned, df_right=products_3_cleaned,
    pairs=star_train_p3, id_column='id', labels=star_train_p3['label']
)

X_train_p3 = train_features_p3.drop(['label', 'id1', 'id2'], axis=1)
y_train_p3 = train_features_p3['label']

clf_p3 = RandomForestClassifier(n_estimators=100, random_state=42)
clf_p3.fit(X_train_p3, y_train_p3)

matcher_p3 = MLBasedMatcher(extractor)
correspondences_ml_p3 = matcher_p3.match(
    df_left=products_1_cleaned, df_right=products_3_cleaned,
    candidates=standard_blocker_p1_p3,
    id_column='id', trained_classifier=clf_p3,
    threshold=0.45, use_probabilities=True
)

results_ml_p3 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_ml_p3, test_pairs=star_val_p3
)
print("--- ML Results for P1 to P3 ---")
display(results_ml_p3)

[INFO ] root - Label distribution: 291 positive, 341 negative
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 812 x 762 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 812 x 762 elements after 0:00:0.123; 173291 blocked pairs (reduction ratio: 0.7199310215533403)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:55.096; found 21831 correspondences.
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  54
[INFO ] root -   True Negatives:  44
[INFO ] root -   False Positives: 4
[INFO ] root -   False Negatives: 8
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.891
[INFO ] root -   Precision: 0.931
[INFO ] root -   Recall:    0.871
[INFO ] root -   F1-Score:  0.900


--- ML Results for P1 to P3 ---


{'precision': 0.9310344827586207,
 'recall': 0.8709677419354839,
 'f1': 0.9,
 'accuracy': 0.8909090909090909,
 'true_positives': 54,
 'false_positives': 4,
 'false_negatives': 8,
 'true_negatives': 44,
 'threshold_used': 0.0,
 'total_correspondences': 21831,
 'filtered_correspondences': 21831,
 'evaluation_timestamp': '2026-04-22T20:47:33.985773'}

In [26]:
# P1 -> P4
star_train_p4 = load_csv(OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" /"prod1_to_prod4_train.csv", add_index=False)
star_val_p4   = load_csv(OUTPUT_DIR / "entity_matching_final_ground_truth" / "per_file_splits" / "prod1_to_prod4_val.csv",   add_index=False)

train_features_p4 = extractor.create_features(
    df_left=products_1_cleaned, df_right=products_4_cleaned,
    pairs=star_train_p4, id_column='id', labels=star_train_p4['label']
)

X_train_p4 = train_features_p4.drop(['label', 'id1', 'id2'], axis=1)
y_train_p4 = train_features_p4['label']

clf_p4 = RandomForestClassifier(n_estimators=100, random_state=42)
clf_p4.fit(X_train_p4, y_train_p4)

matcher_p4 = MLBasedMatcher(extractor)
correspondences_ml_p4 = matcher_p4.match(
    df_left=products_1_cleaned, df_right=products_4_cleaned,
    candidates=standard_blocker_p1_p4,
    id_column='id', trained_classifier=clf_p4,
    threshold=0.45, use_probabilities=True
)

results_ml_p4 = EntityMatchingEvaluator.evaluate_matching(
    correspondences=correspondences_ml_p4, test_pairs=star_val_p4
)
print("--- ML Results for P1 to P4 ---")
display(results_ml_p4)

[INFO ] root - Label distribution: 291 positive, 295 negative
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Starting Entity Matching
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Blocking 812 x 626 elements
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Matching 812 x 626 elements after 0:00:0.108; 143942 blocked pairs (reduction ratio: 0.71682352570862)
[INFO ] PyDI.entitymatching.ml_based.MLBasedMatcher - Entity Matching finished after 0:00:42.046; found 21424 correspondences.
[INFO ] root - Confusion Matrix:
[INFO ] root -   True Positives:  55
[INFO ] root -   True Negatives:  47
[INFO ] root -   False Positives: 7
[INFO ] root -   False Negatives: 7
[INFO ] root - Performance Metrics:
[INFO ] root -   Accuracy:  0.879
[INFO ] root -   Precision: 0.887
[INFO ] root -   Recall:    0.887
[INFO ] root -   F1-Score:  0.887


--- ML Results for P1 to P4 ---


{'precision': 0.8870967741935484,
 'recall': 0.8870967741935484,
 'f1': 0.8870967741935484,
 'accuracy': 0.8793103448275862,
 'true_positives': 55,
 'false_positives': 7,
 'false_negatives': 7,
 'true_negatives': 47,
 'threshold_used': 0.0,
 'total_correspondences': 21424,
 'filtered_correspondences': 21424,
 'evaluation_timestamp': '2026-04-22T20:48:17.908486'}

## Part 3: Data Fusion

### we fist split our data into test and validation 

In [27]:
# Set the base directory to where the notebook is
BASE_DIR = Path(".").resolve()
DATA_DIR = BASE_DIR / "input" / "fusion"

# Load using the short, relative path
val_set = pd.read_csv(DATA_DIR / 'fusion_validation_set.csv')
test_set = pd.read_csv(DATA_DIR / 'fusion_test_set.csv')

print(f"Loaded Validation: {len(val_set)} rows")
print(f"Loaded Test: {len(test_set)} rows")

Loaded Validation: 100 rows
Loaded Test: 100 rows


#### we need to see which file is more trustworthy so we use or fusion validation set to check

In [28]:
# 1. custom matcher as i see that some attribute can be subset of the other between fusion and orignal

def hardware_strict_spec_match(fused_value, expected_value):
    """
    Strips all text and compares only the numbers.
    Prevents 'PCIE x8' from matching 'PCIE x16'.
    """
    if pd.isna(fused_value) or pd.isna(expected_value):
        return False
        
    f = str(fused_value).lower()
    e = str(expected_value).lower()
    
    # Extract all digits (e.g., "PCIE x16" -> ["16"])
    f_numbers = re.findall(r'\d+', f)
    e_numbers = re.findall(r'\d+', e)
    
    # If one has [16] and the other has [8], it's an immediate False.
    if f_numbers != e_numbers:
        return False
    
    # If numbers match (or there are no numbers), do a basic string check
    f_clean = re.sub(r'[^a-z0-9]', '', f)
    e_clean = re.sub(r'[^a-z0-9]', '', e)
    
    return e_clean in f_clean or f_clean in e_clean

# SETUP THE AUDIT STRATEGY
# This tells the evaluator HOW to check for a match in your validation set
audit_strategy = DataFusionStrategy('validation_audit')

# Exact matches for identity
for attr in ['brand', 'product_type', 'model_number']:
    audit_strategy.add_evaluation_function(attr, exact_match)

# Numeric thresholds (15% tolerance) for technical specs
num_attrs = ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s', 'width_mm', 'length_mm', 'height_mm', 'weight_g']
for attr in num_attrs:
    audit_strategy.add_evaluation_function(attr, numeric_tolerance_match, tolerance=0.15)



# Flexible matches for technical strings

# This ensures "x16" passes when compared to "PCIe 3.0 x16"
tech_attrs = ['bus_type', 'interface_type', 'chipset_name',  'storage_connection_type', 'memory_type', 'form_factor']
for attr in tech_attrs:
    audit_strategy.add_evaluation_function(attr, hardware_strict_spec_match)

# tech_attrs = ['chipset_name', 'bus_type', 'interface_type', 'storage_connection_type', 'memory_type', 'form_factor'] #remved title, too messy i think
# for attr in tech_attrs:
#     audit_strategy.add_evaluation_function(attr, tokenized_match)

# 2. INITIALIZE EVALUATOR
evaluator = DataFusionEvaluator(audit_strategy)

# 3. THE AUDIT LOOP
# We run the test on all 4 source files
sources = [products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned]
names = ["p1", "p2", "p3", "p4"]

audit_summary = []

for df_source, name in zip(sources, names):
    # ID Logic: P1 is 'id_left' in your validation set; Others are 'id_right'
    id_col = 'id_left' if name == 'p1' else 'id_right'
    
    # Filter validation set for rows where this source appears
    relevant_gt = val_set[(val_set['source_left'] == name) | (val_set['source_right'] == name)]
    
    # RUN EVALUATION ON THE RAW FILE
    results = evaluator.evaluate(
        fused_df=df_source,
        fused_id_column='id',
        gold_df=relevant_gt,
        gold_id_column=id_col
    )
    
    audit_summary.append({
        'Source': name,
        'Overall_Accuracy': results.get('overall_accuracy'),
        'Details': results.get('per_attribute_scores')
    })

# 4. SHOW THE RANKING
report = pd.DataFrame(audit_summary).set_index('Source')
print("📊 VALIDATION AUDIT RESULTS (Scientific Evidence for Trust)")
display(report[['Overall_Accuracy']])

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'brand'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'product_type'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'model_number'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'vram_gb' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'storage_gb' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'read_speed_mb_s' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'write_speed_mb_s' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'width_mm' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'length_mm' with params {'tolerance': 0.15}
[IN

📊 VALIDATION AUDIT RESULTS (Scientific Evidence for Trust)


,Overall_Accuracy
Source,
p1,0.455550
p2,0.359003
p3,0.332036
p4,0.357834


#### based on trust scores. we use this order of trust

In [29]:
# Set P1 ID as the master ID for the fused records
products_1_cleaned["p1_id"] = products_1_cleaned["id"] 

# Assign trust scores (1 = Lowest, 3 = Highest)
products_1_cleaned.attrs["trust_score"] = 3
products_2_cleaned.attrs["trust_score"] = 2
products_3_cleaned.attrs["trust_score"] = 1
products_4_cleaned.attrs["trust_score"] = 2

# Combine all  REFINED correspondences into one list
all_correspondences = pd.concat([
    correspondences_p1_p2_refined, 
    correspondences_p1_p3_refined, 
    correspondences_p1_p4_refined
], ignore_index=True)

print(f"Total refined links for fusion: {len(all_correspondences):,}")

Total refined links for fusion: 1,976


### Step 1: Define Fusion Strategy

In [30]:
strategy = DataFusionStrategy('hardware_fusion_strategy')

# 1. Identity
for attr in ['brand', 'product_type', 'model_number']:
    strategy.add_attribute_fuser(attr, voting)

# 2. Performance Specs (Numbers)
for attr in ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s']:
    strategy.add_attribute_fuser(attr, minimum) #i think min could be better. lets see

# 3. Technical & Dimensions (Using Trust)
tech_and_dim_attrs = [
    'chipset_name', 'bus_type', 'interface_type', 'storage_connection_type', 
    'memory_type', 'form_factor', 'width_mm', 'length_mm', 'height_mm', 'weight_g'
]
for attr in tech_and_dim_attrs:
    strategy.add_attribute_fuser(attr, prefer_higher_trust, trust_key="trust_score")

# strategy.add_attribute_fuser('title', longest_string)

[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'brand' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'product_type' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'model_number' using rule 'voting'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'vram_gb' using rule 'minimum'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'storage_gb' using rule 'minimum'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'read_speed_mb_s' using rule 'minimum'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'write_speed_mb_s' using rule 'minimum'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'chipset_name' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'bus_type' using rule 'prefer_higher_trust'
[INFO ] PyDI.fusion.strategy - Registered fuser for attribute 'interface_type' using rule 'prefer_higher_tru

### Step 2: Run Fusion

In [31]:
engine = DataFusionEngine(strategy, debug=True, debug_format='json', 
                          debug_file=OUTPUT_DIR / "data_fusion" / "hardware_fusion_debug.jsonl")

fused = engine.run(
    datasets=[products_1_cleaned, products_2_cleaned, products_3_cleaned, products_4_cleaned],
    correspondences=all_correspondences,
    id_column="id",
    include_singletons=False, # Set to True if you want to keep products that didn't have matches
)   

print(f"Final Fused Rows: {len(fused):,}")
display(fused.head(1)) #maybe lower to 1 because the ouput is huge

[INFO ] PyDI.fusion.engine - Fusion debug logging enabled; refer to C:\Users\hussa\OneDrive\Desktop\Mannheim\Semester 1\IE 500 Data Mining\Excercises\00_PythonIntro\hiwi_data_integration\output\data_fusion\hardware_fusion_debug.jsonl for detailed traces.
[INFO ] PyDI.fusion.engine - Starting data fusion with strategy 'hardware_fusion_strategy'
[INFO ] PyDI.fusion.engine - *    Loading correspondences    *
[INFO ] PyDI.fusion.engine - Correspondence ID coverage: matched 2741 of 2741 unique IDs
[INFO ] PyDI.fusion.engine - Created 1036 record groups from 1976 correspondences
[INFO ] PyDI.fusion.engine - Group Size Distribution of 1036 clusters:
[INFO ] PyDI.fusion.engine - 	Cluster Size	| Frequency	| Percentage
[INFO ] PyDI.fusion.engine - 	──────────────────────────────────────────────────
[INFO ] PyDI.fusion.engine - 		2	|	54	|	5.21%
[INFO ] PyDI.fusion.engine - 		3	|	211	|	20.37%
[INFO ] PyDI.fusion.engine - 		4	|	500	|	48.26%
[INFO ] PyDI.fusion.engine - Attribute Consistencies:
[INF

Final Fused Rows: 765


,_id,_fusion_sources,_fusion_source_datasets,brand,bus_type,chipset_name,cluster_id,color,description,form_factor,...,storage_gb,title,title_description,url,vram_gb,weight_g,width_mm,write_speed_mb_s,_fusion_confidence,_fusion_metadata
0,12198483,"[12198483, 19126355, 40122686]","[products_1, products_2, products_3]",Gigabyte,None,GeForce RTX 3080,1002037,None,"CUDA Cores: 8704, Boost Clock: 1800MHz, GDDR6X...",None,...,NaN,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,Gigabyte NVIDIA GeForce RTX 3080 Gaming OC 10G...,https://www.novatech.co.uk/products/gigabyte-n...,10.0,NaN,NaN,NaN,0.333333,"{'_id_rule': 'first_non_null', '_id_inputs': [..."


### Step 3: Evaluate Data Fusion

In [32]:
# 1. Identity (Must be exact)
strategy.add_evaluation_function("brand", exact_match)
strategy.add_evaluation_function("product_type", exact_match)

# 2. Performance & Capacity (Allow 15% tolerance for rounding)
for attr in ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s']:
    strategy.add_evaluation_function(attr, numeric_tolerance_match, tolerance=0.15)

# 3. Dimensions (Allow 15% tolerance)
for attr in ['width_mm', 'length_mm', 'height_mm', 'weight_g']:
    strategy.add_evaluation_function(attr, numeric_tolerance_match, tolerance=0.15)

# 4. Technical Strings (Using Strict Number Matcher)
# This prevents "PCIe x8" from matching "PCIe x16"
for attr in ['chipset_name', 'bus_type', 'interface_type', 'memory_type']:
    strategy.add_evaluation_function(attr, hardware_strict_spec_match)



# Filter for only the verified rows
val_set_filled = val_set[val_set['filled'] == 'y'].copy()
test_set_filled = test_set[test_set['filled'] == 'y'].copy()
#count verified rows
print(f"Verified rows in Validation Set: {len(val_set_filled)}")
print(f"Verified rows in Test Set: {len(test_set_filled)}")

# Ensure numeric columns are actually numeric
for col in ['vram_gb', 'storage_gb', 'read_speed_mb_s', 'write_speed_mb_s']:
    val_set_filled[col] = pd.to_numeric(val_set_filled[col], errors='coerce')



# Initialize the evaluator
evaluator = DataFusionEvaluator(
    strategy, 
    debug=True, 
    debug_file=OUTPUT_DIR / "data_fusion" / "hardware_eval_debug.jsonl", 
    debug_format="json"
)

# Run the evaluation
print("Evaluating hardware fusion results...")
evaluation_results = evaluator.evaluate(
    fused_df=fused,           # Your fused Golden Records
    fused_id_column='p1_id',  # The master ID column you created
    gold_df=test_set_filled,   # val data. will run now with test set after making a few more adjustments. #val_set_filled
    gold_id_column='id_left'  # The P1 ID column in the validation set
)

# Display results
print("\nHardware Fusion Evaluation Results:")
print("=" * 40)
for metric, value in evaluation_results.items():
    if isinstance(value, float):
        print(f"  {metric}: {value:.3f}")
    else:
        print(f"  {metric}: {value}")
        
print(f"\nOverall Accuracy: {evaluation_results.get('overall_accuracy', 0):.1%}")    

[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'brand'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'product_type'
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'vram_gb' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'storage_gb' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'read_speed_mb_s' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'write_speed_mb_s' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'width_mm' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'length_mm' with params {'tolerance': 0.15}
[INFO ] PyDI.fusion.strategy - Registered evaluation function for attribute 'height_mm' with p

Verified rows in Validation Set: 100
Verified rows in Test Set: 100
Evaluating hardware fusion results...


[INFO ] PyDI.fusion.evaluation - Evaluation complete: 0.402 overall accuracy (878/2185)
[INFO ] PyDI.fusion.evaluation - Evaluation mismatches by attribute (debug): 1307 total
[INFO ] PyDI.fusion.evaluation - 	Attribute                        |  Errors | Percentage
[INFO ] PyDI.fusion.evaluation - 	───────────────────────────────────────────────────────
[INFO ] PyDI.fusion.evaluation - 	length_mm                        |      90 |      6.89%%
[INFO ] PyDI.fusion.evaluation - 	height_mm                        |      90 |      6.89%%
[INFO ] PyDI.fusion.evaluation - 	width_mm                         |      90 |      6.89%%
[INFO ] PyDI.fusion.evaluation - 	title                            |      86 |      6.58%%
[INFO ] PyDI.fusion.evaluation - 	weight_g                         |      80 |      6.12%%
[INFO ] PyDI.fusion.evaluation - 	description                      |      80 |      6.12%%
[INFO ] PyDI.fusion.evaluation - 	title_description                |      79 |      6.04%%
[INFO ]


Hardware Fusion Evaluation Results:
  overall_accuracy: 0.402
  macro_accuracy: 0.437
  num_evaluated_records: 100
  num_evaluated_attributes: 26
  total_evaluations: 2185
  total_correct: 878
  bus_type_accuracy: 0.550
  bus_type_count: 100
  product_type_accuracy: 0.940
  product_type_count: 100
  weight_g_accuracy: 0.070
  weight_g_count: 86
  interface_type_accuracy: 0.220
  interface_type_count: 100
  memory_type_accuracy: 0.719
  memory_type_count: 32
  model_accuracy: 0.263
  model_count: 99
  title_accuracy: 0.140
  title_count: 100
  brand_accuracy: 0.910
  brand_count: 100
  vram_gb_accuracy: 0.800
  vram_gb_count: 25
  description_accuracy: 0.200
  description_count: 100
  cluster_id_accuracy: 0.710
  cluster_id_count: 100
  price_accuracy: 0.310
  price_count: 100
  model_number_accuracy: 0.556
  model_number_count: 99
  chipset_name_accuracy: 0.920
  chipset_name_count: 25
  storage_connection_type_accuracy: 0.431
  storage_connection_type_count: 72
  length_mm_accuracy: 